# Fixture-level ensemble model

Build one row per player-fixture in 2025/26 and evaluate chronological folds across the season. Every fixture in a gameweek uses form calculated strictly from earlier gameweeks.


In [1]:
import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

from fantasy_football.model.base_models import (
    load_inseason_scores,
    load_preseason_scores,
)
from fantasy_football.model.ensemble import (
    DEFAULT_ENSEMBLE_ARTIFACT_PATH,
    ENSEMBLE_CATEGORICAL_FEATURES,
    ENSEMBLE_FEATURES,
)
from fantasy_football.model.season_context import get_promoted_teams
from fantasy_football.model.training.utils import (
    load_gw_data,
    load_player_data,
    load_team_data,
)

RANDOM_STATE = 42



## Leakage-free base-model scores

The base-model notebooks own their historical training periods. This notebook loads their exported scores/models and only builds ensemble rows from the 2025/26 fixture data.


In [2]:
season_2025_26 = (
    load_gw_data("2025-26")
    .drop_duplicates(["element", "fixture"], keep="last")
    .reset_index(drop=True)
)
print("2025/26:", season_2025_26.shape)


2025/26: (29747, 46)


In [3]:
preseason_scores = load_preseason_scores()
preseason_scores.head()


,code,name,preseason_model_score
0,622758,Aaron Anselmino,0.149392
1,55459,Aaron Cresswell,1.291941
2,472713,Aaron Hickey,0.251682
3,225321,Aaron Ramsdale,2.213535
4,214590,Aaron Wan-Bissaka,2.354585


In [4]:
inseason_scores = load_inseason_scores()
inseason_scores.head()


,element,fixture,GW,name,inseason_model_score
0,1,9,1,David Raya Martín,1.158351
1,1,11,2,David Raya Martín,3.239340
2,1,25,3,David Raya Martín,3.352658
3,1,31,4,David Raya Martín,3.255965
4,1,41,5,David Raya Martín,3.365063


## Ensemble dataset


In [5]:
team_lookup = load_team_data("2025-26").set_index("id")["name"]
player_codes = load_player_data("2025-26")[["id", "code"]].rename(columns={"id": "element"})
ensemble_data = season_2025_26.merge(
    inseason_scores[["element", "fixture", "inseason_model_score"]],
    on=["element", "fixture"], how="left", validate="one_to_one",
)
ensemble_data = ensemble_data.merge(player_codes, on="element", how="left")
ensemble_data = ensemble_data.merge(
    preseason_scores[["code", "preseason_model_score"]], on="code", how="left"
)
ensemble_data["opponent"] = ensemble_data["opponent_team"].map(team_lookup)
ensemble_data["fixtures_in_gameweek"] = ensemble_data.groupby(
    ["element", "GW"]
)["fixture"].transform("nunique")
ensemble_data["is_double_gameweek"] = ensemble_data["fixtures_in_gameweek"].gt(1).astype(int)
ensemble_data["was_home"] = ensemble_data["was_home"].astype(int)
promoted_teams = get_promoted_teams("2025-26")
ensemble_data["team_promoted"] = ensemble_data["team"].isin(promoted_teams).astype(int)
ensemble_data["opponent_promoted"] = ensemble_data["opponent"].isin(promoted_teams).astype(int)

ensemble_features = list(ENSEMBLE_FEATURES)
categorical_features = list(ENSEMBLE_CATEGORICAL_FEATURES)
target = "total_points"
ensemble_data[categorical_features] = ensemble_data[categorical_features].fillna("__MISSING__").astype(str)
assert not ensemble_data.duplicated(["element", "fixture"]).any()
assert ensemble_data["opponent"].ne("__MISSING__").all()
print(f"Fixture rows: {len(ensemble_data):,}")
ensemble_data[["name", "fixture", *ensemble_features, target]].head()


Fixture rows: 29,747


,name,fixture,GW,preseason_model_score,inseason_model_score,position,team,opponent,was_home,is_double_gameweek,team_promoted,opponent_promoted,total_points
0,Reinildo Mandava,5,1,NaN,1.158351,DEF,Sunderland,West Ham,1,0,1,0,6
1,Lewis Dobbin,2,1,0.501644,1.158351,MID,Aston Villa,Newcastle,1,0,0,0,0
2,Ryan Christie,1,1,2.339665,1.158351,MID,Bournemouth,Liverpool,0,0,0,0,0
3,Zeki Amdouni,6,1,NaN,1.158351,FWD,Burnley,Spurs,0,0,1,0,0
4,Lucas Tolentino Coelho de Lima,5,1,2.228029,1.158351,MID,West Ham,Sunderland,0,0,0,1,2


In [6]:
def fit_ensemble_model(training_data):
    model = CatBoostRegressor(
        iterations=250, learning_rate=0.03, depth=6, loss_function="RMSE",
        random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False,
    )
    model.fit(
        training_data[ensemble_features], training_data[target],
        cat_features=categorical_features,
    )
    return model


def evaluate(validation, predictions, top_fraction=0.10):
    scored = validation[["GW", target]].copy()
    scored["prediction"] = predictions
    scored["absolute_error"] = (scored[target] - scored["prediction"]).abs()
    top_end_by_gameweek = []
    for _, gameweek in scored.groupby("GW"):
        n = max(1, int(np.ceil(len(gameweek) * top_fraction)))
        predicted_top = gameweek.nlargest(n, "prediction")
        actual_top_indices = set(gameweek.nlargest(n, target).index)
        top_end_by_gameweek.append({
            "top_decile_avg_actual_points": predicted_top[target].mean(),
            "top_decile_hit_rate": predicted_top.index.isin(actual_top_indices).mean(),
            "top_decile_oracle_regret": (
                gameweek.nlargest(n, target)[target].mean()
                - predicted_top[target].mean()
            ),
        })
    top_end = pd.DataFrame(top_end_by_gameweek).mean()
    return {
        "num": len(scored),
        "MAE": mean_absolute_error(scored[target], scored["prediction"]),
        "RMSE": root_mean_squared_error(scored[target], scored["prediction"]),
        "R2": r2_score(scored[target], scored["prediction"]),
        **top_end.to_dict(),
        **{
            f"absolute_error_p{percentile}": scored["absolute_error"].quantile(percentile / 100)
            for percentile in (25, 50, 75, 90)
        },
    }


## Single late-season holdout: GW30+

Train on GW1–29 and evaluate every fixture from GW30 onward. This provides one conventional holdout alongside the phase-by-phase walk-forward results below.


In [7]:
holdout_train = ensemble_data.loc[ensemble_data["GW"].lt(30)]
holdout_validation = ensemble_data.loc[ensemble_data["GW"].ge(30)]
holdout_model = fit_ensemble_model(holdout_train)
holdout_predictions = holdout_model.predict(holdout_validation[ensemble_features])
holdout_results = pd.DataFrame([
    {"train_gws": "1-29", "validation_gws": "30-38",
     **evaluate(holdout_validation, holdout_predictions)}
])
holdout_results.T.round(3)


,0
train_gws,1-29
validation_gws,30-38
num,7404
MAE,0.958169
RMSE,1.873555
R2,0.32077
top_decile_avg_actual_points,3.689771
top_decile_hit_rate,0.359078
top_decile_oracle_regret,3.19161
absolute_error_p25,0.040567


## Chronological phase validation

Every fold trains on earlier gameweeks only. GW1–5 provide the minimum fitting history, so honest validation of GW1–5 itself would require ensemble training data from a previous season.


In [8]:
folds = [
    ("early", 5, 6, 10),
    ("early-mid", 10, 11, 15),
    ("mid", 19, 20, 24),
    ("late-mid", 28, 29, 33),
    ("late", 33, 34, 38),
]

results = []
validation_predictions = []
for phase, train_end, valid_start, valid_end in folds:
    train = ensemble_data.loc[ensemble_data["GW"].le(train_end)]
    validation = ensemble_data.loc[ensemble_data["GW"].between(valid_start, valid_end)]
    model = fit_ensemble_model(train)
    predictions = model.predict(validation[ensemble_features])
    results.append({
        "phase": phase, "train_through": train_end,
        "validation_gws": f"{valid_start}-{valid_end}",
        **evaluate(validation, predictions),
    })
    validation_predictions.append(
        validation[["element", "fixture", "GW", target]].assign(phase=phase, prediction=predictions)
    )

fold_results = pd.DataFrame(results)
fold_results.round(3)


,phase,train_through,validation_gws,num,MAE,RMSE,R2,top_decile_avg_actual_points,top_decile_hit_rate,top_decile_oracle_regret,absolute_error_p25,absolute_error_p50,absolute_error_p75,absolute_error_p90
0,early,5,6-10,3723,1.007,1.957,0.327,3.613,0.323,3.573,0.061,0.359,1.274,2.653
1,early-mid,10,11-15,3779,1.031,2.044,0.317,4.026,0.363,3.445,0.043,0.298,1.323,2.781
2,mid,19,20-24,3998,1.025,1.885,0.317,3.696,0.356,3.288,0.063,0.322,1.429,2.681
3,late-mid,28,29-33,4209,0.971,1.913,0.324,3.688,0.346,3.365,0.032,0.237,1.327,2.645
4,late,33,34-38,4015,0.958,1.839,0.318,3.590,0.353,3.160,0.044,0.224,1.287,2.600


In [9]:
out_of_time_predictions = pd.concat(validation_predictions, ignore_index=True)
pd.Series(
    evaluate(out_of_time_predictions, out_of_time_predictions["prediction"]),
    name="all chronological folds",
).round(3)


num                             19724.000
MAE                                 0.998
RMSE                                1.927
R2                                  0.321
top_decile_avg_actual_points        3.723
top_decile_hit_rate                 0.348
top_decile_oracle_regret            3.366
absolute_error_p25                  0.048
absolute_error_p50                  0.286
absolute_error_p75                  1.326
absolute_error_p90                  2.673
Name: all chronological folds, dtype: float64

## Production ensemble artifact

After validation, refit the same ensemble procedure on every available 2025/26 fixture row. This artifact is separate from the fold models above and is used only to forecast later seasons.


In [10]:
final_ensemble_model = fit_ensemble_model(ensemble_data)
DEFAULT_ENSEMBLE_ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "model": final_ensemble_model,
        "feature_columns": ensemble_features,
        "categorical_columns": categorical_features,
        "training_season": "2025-26",
    },
    DEFAULT_ENSEMBLE_ARTIFACT_PATH,
)
print(f"Saved production ensemble model to {DEFAULT_ENSEMBLE_ARTIFACT_PATH}")


Saved production ensemble model to /Users/calumthompson/Documents/fantasy_football_v2/fantasy_football/model/artifacts/ensemble_model.joblib
